# Fetch data

In [19]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
y = y.iloc[:, 0]

# Prepare attributes, pipeline and split sets

In [20]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

# NaNN and NaN --> np.nan
X = X.replace(["NaN", "NaNN", "", " "], np.nan)

# Change categorical variables to numeric
cols_to_numeric = ["Cholesterol", "Copper", "Tryglicerides", "Platelets"]
X[cols_to_numeric] = X[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Change Stage to category
X["Stage"] = X["Stage"].astype("category")

# Split data into training and test sets
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

# Cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

# Preprocessor
cat_cols = X_rest.select_dtypes(include=["object", "str", "category"]).columns
num_cols = X_rest.select_dtypes(include=["number"]).columns

# Preprocessor with imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ]
)

# Bayes classificator

In [21]:

from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

model = Pipeline([
    ("prep", preprocessor),
    ("nb", GaussianNB())
])

scores = cross_val_score(model, X_rest, y_rest, cv=cv, scoring="accuracy")
print(f"Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")
    
print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model.fit(X_rest, y_rest)
y_pred = model.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred))


Average accuracy scores for each CV fold:

	Fold 1: 0.6176
	Fold 2: 0.6471
	Fold 3: 0.6765
	Fold 4: 0.6471
	Fold 5: 0.7273
	Fold 6: 0.7576
	Fold 7: 0.6061
	Fold 8: 0.5455
	Fold 9: 0.6970
	Fold 10: 0.6970

Average Accuracy from CV: 0.6619

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.77      0.79      0.78        47
          CL       0.12      0.40      0.19         5
           D       0.85      0.53      0.65        32

    accuracy                           0.67        84
   macro avg       0.58      0.57      0.54        84
weighted avg       0.76      0.67      0.70        84



In [22]:
from sklearn.tree import DecisionTreeClassifier


model = Pipeline(
    [
        ("prep", preprocessor),
        ("dt", DecisionTreeClassifier(random_state=67)),
    ]
)

scores = cross_val_score(model, X_rest, y_rest, cv=cv, scoring="accuracy")

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model.fit(X_rest, y_rest)
y_pred = model.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred))

Average accuracy scores for each CV fold:

	Fold 1: 0.5588
	Fold 2: 0.6471
	Fold 3: 0.4118
	Fold 4: 0.6471
	Fold 5: 0.6970
	Fold 6: 0.6667
	Fold 7: 0.6667
	Fold 8: 0.6364
	Fold 9: 0.7576
	Fold 10: 0.7576

Average Accuracy from CV: 0.6447

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.65      0.72      0.69        47
          CL       0.20      0.20      0.20         5
           D       0.56      0.47      0.51        32

    accuracy                           0.60        84
   macro avg       0.47      0.46      0.47        84
weighted avg       0.59      0.60      0.59        84



In [23]:
from sklearn.ensemble import RandomForestClassifier

model_rf = Pipeline(
    [
        ("prep", preprocessor),
        ("rf", RandomForestClassifier(random_state=67)),
    ]
)

scores = cross_val_score(
    model_rf, X_rest, y_rest, cv=cv, scoring="accuracy"
)

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model_rf.fit(X_rest, y_rest)
y_pred = model_rf.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred, zero_division=0))

Average accuracy scores for each CV fold:

	Fold 1: 0.7647
	Fold 2: 0.7941
	Fold 3: 0.7647
	Fold 4: 0.8824
	Fold 5: 0.7879
	Fold 6: 0.7273
	Fold 7: 0.6667
	Fold 8: 0.6061
	Fold 9: 0.7879
	Fold 10: 0.8485

Average Accuracy from CV: 0.7630

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.73      0.87      0.80        47
          CL       0.00      0.00      0.00         5
           D       0.68      0.59      0.63        32

    accuracy                           0.71        84
   macro avg       0.47      0.49      0.48        84
weighted avg       0.67      0.71      0.69        84



In [24]:
from sklearn.svm import SVC

model_svm = Pipeline(
    [
        ("prep", preprocessor),
        ("svm", SVC(random_state=67)),
    ]
)

scores = cross_val_score(
    model_svm, X_rest, y_rest, cv=cv, scoring="accuracy"
)

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model_svm.fit(X_rest, y_rest)
y_pred = model_svm.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred, zero_division=0))

Average accuracy scores for each CV fold:

	Fold 1: 0.5882
	Fold 2: 0.5588
	Fold 3: 0.5882
	Fold 4: 0.5294
	Fold 5: 0.5455
	Fold 6: 0.6364
	Fold 7: 0.6061
	Fold 8: 0.5758
	Fold 9: 0.6061
	Fold 10: 0.5758

Average Accuracy from CV: 0.5810

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.56      0.96      0.70        47
          CL       0.00      0.00      0.00         5
           D       0.33      0.03      0.06        32

    accuracy                           0.55        84
   macro avg       0.30      0.33      0.25        84
weighted avg       0.44      0.55      0.42        84

